In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [3]:
llm = ChatOpenAI(model = 'gpt-3.5-turbo')

In [16]:
class ProductReview(BaseModel):
    customer_name:str = Field(description = "Full name of the customer providing the review")
    product_id:str = Field(description = 'Unique product identifier')
    rating:int = Field(gt = 0, lt = 6, description = 'Rating between 1 and 5')
    review:str = Field(default = "", description = 'Feedback by the customer')
    would_recommend:bool = Field(default = False, description = 'Whether the customer would recommend the product')
    sentiment:Literal["Positive", 'Negative', "Neutral"] = Field(description = "Overall sentiment of the customer") #after reading the review, llm provide sentiment output here

In [17]:
parser = PydanticOutputParser(pydantic_object = ProductReview)

In [18]:
template = PromptTemplate(template = "I have the following customer review \n{text} \n {format}",
                            input_variables = ['text'],
                            partial_variables = {'format':parser.get_format_instructions()})

In [19]:
review = """
            Hi, this is Zira. I bought the product id abc. I would give it 4 stars. 
            Great value for money. Delivery was prompt.
            I would recommend it to others
"""

In [20]:
prompt = template.invoke({'text':review})
print(prompt)

text='I have the following customer review \n\n            Hi, this is Zira. I bought the product id abc. I would give it 4 stars. \n            Great value for money. Delivery was prompt.\n            I would recommend it to others\n \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"customer_name": {"description": "Full name of the customer providing the review", "title": "Customer Name", "type": "string"}, "product_id": {"description": "Unique product identifier", "title": "Product Id", "type": "string"}, "rating": {"description": "Rating between 1 and 5", "exclus

In [21]:
result = llm.invoke(prompt)

In [22]:
print(result)

content='{\n  "customer_name": "Zira",\n  "product_id": "abc",\n  "rating": 4,\n  "review": "Great value for money. Delivery was prompt.",\n  "would_recommend": true,\n  "sentiment": "Positive"\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 393, 'total_tokens': 448, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DI9ktOogiiX2PtByTvsIKR1IbnUGs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cdc22-88ad-78c1-ae83-a0d3ba753279-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 393, 'output_tokens': 55, 'total_tokens': 448, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

In [23]:
final_result = parser.invoke(result)
print(final_result)

customer_name='Zira' product_id='abc' rating=4 review='Great value for money. Delivery was prompt.' would_recommend=True sentiment='Positive'


In [24]:
chain = template | llm | parser
result = chain.invoke({'text': review})

In [25]:
print(result)

customer_name='Zira' product_id='abc' rating=4 review='Great value for money. Delivery was prompt.' would_recommend=True sentiment='Positive'
